In [2]:
%pip install -U langchain-core

  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/573.8 kB ? eta -:--:--
   ---------------------------------------- 573.8/573.8 kB 6.0 MB/s  0:00:00
   ---------------------------------------- 0.0/869.0 kB ? eta -:--:--
   ---------------------------------------- 869.0/869.0 kB 6.0 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 10.3 MB/s  0:00:00
Using cached t


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import langchain_core

print("LangChain Core installed successfully!")
print("Version:", langchain_core.__version__)

LangChain Core installed successfully!
Version: 1.6.12


In [5]:
from langchain_core.runnables import RunnableLambda

def make_greeting(data:dict) -> str:
    return f"Hello, {data['name']}!"
greeting=RunnableLambda(make_greeting)

print(greeting.invoke({"name":"harshita"}))

Hello, harshita!


### Batch,Async and streaming examples 

In [13]:
from langchain_core.runnables import RunnableLambda
double=RunnableLambda(lambda x:x*2)
# print(double.invoke(4))
# print(double.batch([1,2,3]))

# In async fucntion :
# result=await double.ainvoke(4)
# print(result)
# result=await double.abatch([1,2,3])
# print(result)

# streaming depends on the Runnables.
for chunk in double.stream(5):
    print(chunk)

10


In [ ]:
# aync function is a coroutine function and await keyword marks an operation that fucntion needs to wait for .
# while the operation is waiting the puthon asynchronous event loop can run other eligible task instaed f leaving the entire event loop idle 

### RunnableConfig

In [17]:
from langchain_core.runnables import RunnableLambda 
result=greeting.invoke(
    {'name':'harshita'},
    config={
        "run_name":"greeting_step",
        "tags":['demo','runnable'],
        "metadata":{"features":"welcome"}
    }
)
print(result)

Hello, harshita!


In [18]:
from langchain_core.runnables import RunnableLambda,RunnableSequence

clean=RunnableLambda(lambda text:text.strip())
mark_upper=RunnableLambda(lambda text:text.upper())

add_mark=RunnableLambda(lambda text:f"Result: {text}")

chain=RunnableSequence(clean,mark_upper,add_mark)

print(chain.invoke("  hello world   "))


Result: HELLO WORLD


# RUNNABLE SEQUNCE

####GENERATE A JOKE THEN EXPLAIN IT 

In [ ]:
from langchain_core.runnables import RunnableLambda

joke_generator = RunnableLambda(
    lambda data: f"Why did {data['topic']} cross the road? To get to the other side."
)
joke_explainer = RunnableLambda(
    lambda joke: f"Explanation: this is a classic wordplay joke. Original: {joke}"
)

joke_chain = joke_generator | joke_explainer
print(joke_chain.invoke({"topic": "the robot"}))

# RUNNABLE PARALLEL 

In [ ]:
from langchain_core.runnables import RunnableLambda,RunnableParallel

tweet=RunnableLambda(lambda x:f"short post about{x['topic']}")
linkedin=RunnableLambda(lambda x:f"Professional post about {x['topic']}")
keywords=RunnableLambda(lambda x:[x['topic'],"AI","Learning"])

content_chain=RunnableParallel({
    "tweet":tweet,
    "linkedin":linkedin,
    "keywords":keywords
})

result=content_chain.invoke({"topic":"Runnables"})
print(result["tweet"])
print(result["linkedin"])
print(result["keywords"])

# SEQUENTIAL + PARALLEL 

In [ ]:
from langchain_core.runnables import RunnableLambda,RunnablePrallel

prepare=RunnableLamda(lamda x:{"topic":x["topic"].strip().title()})
make_tweet=RunnableLambda(lambda x:f"Quick thought : {x['topic']}") 
make_outline=RunnableLambda((lambda x:[f"intro to {x['topic']}","key idea" ,"conclusion"])) 

branches =RunnablePrallel({
    "tweet":make_tweet,
    "outline":make_outline,})

chain = prepare | branches
print(chain.invoke({"topic": "  runnable design "}))

## RunnablePassthrough and assign()

In [ ]:
from langchain_core.runnables import RunnablePassthrough

identity = RunnablePassthrough()
print(identity.invoke({"question": "What is LCEL?", "level": "beginner"}))
# Same dictionary comes back

##  RunnablePassthrough.assign()

from langchain_core.runnables import RunnableLambda, RunnablePassthrough

add_length = RunnableLambda(lambda data: len(data["text"]))
chain = RunnablePassthrough.assign(character_count=add_length)

print(chain.invoke({"text": "Hello", "source": "user"}))
# {'text': 'Hello', 'source': 'user', 'character_count': 5}

### Preserve both an original result and a derived result

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

make_joke = RunnableLambda(lambda x: f"Why did {x['topic']} learn Python? For better logic.")
explain = RunnableLambda(lambda data: f"Explanation of: {data['joke']}")

chain = (
    RunnablePassthrough.assign(joke=make_joke)
    | RunnablePassthrough.assign(explanation=explain)
)

print(chain.invoke({"topic": "a robot"}))
# Output dictionary contains topic, joke, and explanation

## RunnableLambda

In [19]:
from langchain_core.runnables import RunnableLambda

def word_count(data):
    return {"word_count":len(data["text"].split())}

count_words=RunnableLambda(word_count)
print(count_words.invoke({"text":"Runnables make pipleine easier"}))

{'word_count': 4}


In [20]:
uppercase = RunnableLambda(lambda text: text.upper())
print(uppercase.invoke("langchain"))

LANGCHAIN


In [21]:
def safe_word_count(data):
    text = data.get("text", "")
    if not isinstance(text, str):
        raise TypeError("'text' must be a string")
    return {"word_count": len(text.split())}

safe_count = RunnableLambda(safe_word_count)
print(safe_count.invoke({"text": "check your inputs"}))

{'word_count': 3}


In [ ]:
from langchain_core.runnables import RunnableLambda,RunnableBranch 

short_summary=RunnableLambda(lambda x:"Short summary: " + x["text"][:80])
long_summary=RunnableLambda(lambda x:"Long summary requested for : "+x["text"])
default_reply=RunnableLambda(lambda x :"Please provide text.")

branch=RunnableBranch(
    (lambda x:len(x.get("text",""))>300,long_summary),
    (lambda x:0<len(x.get("text",""))<300,short_summary),
    default_reply,
)
print(branch.invoke({"text": "A short article."}))
print(branch.invoke({"text": "x" * 500}))
print(branch.invoke({"text": ""}))

### Example: route support messages

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableBranch

billing = RunnableLambda(lambda x: "Route to billing support")
technical = RunnableLambda(lambda x: "Route to technical support")
general = RunnableLambda(lambda x: "Route to general support")

router = RunnableBranch(
    (lambda x: "refund" in x["message"].lower()
              or "payment" in x["message"].lower(), billing),
    (lambda x: "error" in x["message"].lower()
              or "bug" in x["message"].lower(), technical),
    general,
)

print(router.invoke({"message": "I need a refund"}))

## Practical LLM example (optional API key)

In [ ]:
# Install (example)
# pip install -U langchain-core langchain-openai python-dotenv

# .env file:
# OPENAI_API_KEY=your_key_here

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} to a beginner in 3 bullet points."
)
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
parser = StrOutputParser()

chain = prompt | model | parser
answer = chain.invoke({"topic": "LangChain Runnables"})

### Simple RAG-style data flow

In [ ]:
# Conceptual LCEL pattern; retriever and prompt must be configured first.
# rag_chain = (
#     {"context": retriever, "question": RunnablePassthrough()}
#     | prompt
#     | model
#     | StrOutputParser()
# )
# answer = rag_chain.invoke("What does the policy say about refunds?")

### Complete runnable-only practice file

In [ ]:
from langchain_core.runnables import (
    RunnableLambda,
    RunnableSequence,
    RunnableParallel,
    RunnablePassthrough,
    RunnableBranch,
)

# 1) Sequential workflow
clean = RunnableLambda(lambda x: x.strip())
upper = RunnableLambda(lambda x: x.upper())
sequence = clean | upper
print("Sequence:", sequence.invoke("  hello runnables  "))

# 2) Parallel workflow
parallel = RunnableParallel({
    "uppercase": RunnableLambda(lambda x: x.upper()),
    "length": RunnableLambda(lambda x: len(x)),
})
print("Parallel:", parallel.invoke("LangChain"))

# 3) Preserve input and add a computed field
add_length = RunnablePassthrough.assign(
    length=RunnableLambda(lambda d: len(d["text"]))
)
print("Assign:", add_length.invoke({"text": "Hello", "source": "demo"}))

# 4) Custom Python logic
word_count = RunnableLambda(lambda d: len(d["text"].split()))
print("Word count:", word_count.invoke({"text": "one two three"}))

# 5) Conditional routing
router = RunnableBranch(
    (lambda d: d["score"] >= 80,
     RunnableLambda(lambda d: "High score")),
    (lambda d: d["score"] >= 50,
     RunnableLambda(lambda d: "Pass")),
    RunnableLambda(lambda d: "Needs improvement"),
)
print("Branch:", router.invoke({"score": 72}))

# 6) Batch: same Runnable, many inputs
print("Batch:", upper.batch(["one", "two", "three"]))

# 7) A chain that keeps original text and adds a transformation
pipeline = RunnablePassthrough.assign(
    cleaned=RunnableLambda(lambda d: d["text"].strip()),
).assign(
    uppercase=RunnableLambda(lambda d: d["cleaned"].upper())
)
print("Combined:", pipeline.invoke({"text": "  hello  "}))